In [ ]:
import dotenv
import os
dotenv.load_dotenv()
key = os.getenv("OPENAI_API_KEY")
url = os.getenv("OPENAI_API_URL")
app_id = os.getenv("OPENAI_APP_ID")
user_id = os.getenv("OPENAI_USER_ID")
company_id = os.getenv("OPENAI_COMPANY_ID")
api_version = os.getenv("OPENAI_API_VERSION")

In [ ]:
user_prompt = "Respond with one word, with a player's name: Who is the GOAT of football? Name one player"

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=key,
    base_url=url,
    default_headers={
        "x-app-id": app_id,
        "x-user-id": user_id,
        "x-company-id": company_id,
        "x-api-version": api_version
    }
)

response = client.chat.completions.create(
      model="AZURE_GPT_4o_2024_1120",
      messages=[
          {
              "role": "system",
              "content": "You are a helpful assistant that answers questions based on the information that is publicly available. Keep answers short and on-point. If the answer cannot be determined, say 'No relevant information.'"
          },
          {
              "role": "user",
              "content": f"""
                  Question:
                  {user_prompt}
                  """
          }
                  ],
      temperature=0.25
      )

print(response.choices[0].message.content)

In [ ]:
import pandas as pd
ticker = 'GM'
df = pd.read_json('/home/veselin/Documents/Programiranje/edgar-insight-rag/data/chunks/' + ticker + '/2025-10-K.chunks.jsonl', lines=True)

In [ ]:
df = df[df["content_type"] == "table"]

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
df.columns

In [ ]:
import pandas as pd


def dataframe_from_logical_table(headers, rows):
    if not isinstance(headers, list) or not isinstance(rows, list):
        raise TypeError("Logical table fields must remain native JSON arrays.")
    if not all(isinstance(values, list) for values in rows):
        raise TypeError("Every logical row must remain a native JSON array.")
    if any(len(values) != len(headers) for values in rows):
        raise ValueError("Logical rows do not match logical column headers.")
    display_headers = [value or f"Column {index + 1}" for index, value in enumerate(headers)]
    return pd.DataFrame(rows, columns=display_headers)


def dataframes_from_table_chunk(row):
    if row["content_type"] != "table":
        raise ValueError("Select a table chunk before building a table dataframe.")

    if row["composition_mode"] == "compound":
        fragments = row["logical_fragments"]
        return [
            dataframe_from_logical_table(
                fragment["logical_column_headers"], fragment["logical_rows"]
            )
            for fragment in fragments
        ]

    headers = row["logical_column_headers"]
    rows = row["logical_rows"]
    return [dataframe_from_logical_table(headers, rows)]


table_chunks = df[df["content_type"] == "table"]
clean_tables = dataframes_from_table_chunk(table_chunks.iloc[0])
for clean_df in clean_tables:
    print(clean_df.to_string(index=False))


In [ ]:
table_chunks.head(1)[["logical_column_headers", "logical_rows", "composition_mode"]]

In [ ]:
df.to_json('GM_tables.jsonl', orient='records', lines=True)